In [98]:
import pandas as pd
import numpy as np
import io

In [99]:
# ==== File Paths ====
duelist_file = r"Input_Files/duelist_dump_march_10.xlsx"
duelist_main_file = r"Z:/1.Reports Repository/FY 2082.83/1. Duelist/8.Fagun/Duelist 9th March, 2026.xlsb"
insurance = r"Z:/1.Reports Repository/FY 2082.83/1. Duelist/8.Fagun/Fagun insurance.xlsx"
output_file = r"Output_Files/updated_duelist.xlsx"

insurance = pd.read_excel(insurance, dtype={"MainCode": str})

if duelist_file.endswith(".xlsb"):
    duelist = pd.read_excel(
        duelist_file,
        dtype={"MainCode": str, "AcCodeForChg": str, "Nominee": str},
        engine="pyxlsb"
)
else:
    duelist = pd.read_excel(
        duelist_file,
        dtype={"MainCode": str, "AcCodeForChg": str, "Nominee": str},
        engine="openpyxl"
    )  

if duelist_main_file.endswith(".xlsb"):
    duelist_main = pd.read_excel(
        duelist_main_file,
        dtype={"MainCode": str, "AcCodeForChg": str, "Nominee": str},
        engine="pyxlsb"
    )
else:
    duelist_main = pd.read_excel(
        duelist_main_file,
        dtype={"MainCode": str, "AcCodeForChg": str, "Nominee": str},
        engine="openpyxl"
    )

print(len(duelist))
duelist.head(2)

26735


,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,Address,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
0,001,20,HP-3WH (TEMPO),01,NaN,00024235,NaN,00102000024235000002,BISHONATH MAHATO,SAKHUWA PRASAUNI SAKHUWA PRASAUNI PARSA,...,FOTON OMEGA STREAM,NaN,40% Downpayment,5 FOTON,NaN,NaN,NaN,025,17.0,17/01/2026
1,001,20,HP-3WH (TEMPO),01,~~~~~,~~~~~,~~,~~,~~GROUP TOTAL 1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [100]:
def clean_columns(df):
    df.columns = df.columns.str.strip()
    return df

duelist = clean_columns(duelist)
duelist_main = clean_columns(duelist_main)
insurance = clean_columns(insurance)

In [101]:
insurance.dtypes

BBB                          int64
AT                          object
AcTypeDesc                  object
ClientCode                   int64
MainCode                    object
Name                        object
InsPolicyNo                 object
InsIssueDate                object
InsMaturityDate             object
date                datetime64[ns]
InsuredAmt                 float64
InsPremium                 float64
InsurancePremium           float64
Insurer                     object
MortgageCode                object
Identifier                  object
Phone                        int64
MobileNo                   float64
Mobile1                    float64
dtype: object

In [102]:
insurance["date"] = pd.to_datetime(insurance["date"])

In [103]:
yesterday = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)

insurance["InsurancePremium"] = np.where(
    yesterday < insurance["date"],
    insurance["InsPremium"],
    0
)

In [104]:
duelist = duelist.copy()
duelist = duelist[(duelist['ClientCode']!="~~~~~")]
print(len(duelist))
duelist.head(2)

26680


,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,Address,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
0,001,20,HP-3WH (TEMPO),01,NaN,00024235,NaN,00102000024235000002,BISHONATH MAHATO,SAKHUWA PRASAUNI SAKHUWA PRASAUNI PARSA,...,FOTON OMEGA STREAM,NaN,40% Downpayment,5 FOTON,NaN,NaN,NaN,025,17.0,17/01/2026
2,001,21,HP-2WH NEPAL ARMY SCHEME,01,NaN,00010670,NaN,00102100010670000006,SAIRA SHAKYA,GUITOLE LALITPUR,...,Ray ZR Disc Hybrid 125 Street Rally OBII,NaN,30% Downpayment,2 SCOOTER,NaN,CENTRAL,NaN,029,15.0,21/07/2025


In [105]:
print(len(duelist))
print(duelist["MainCode"].nunique())
print(duelist["MainCode"].duplicated().sum())
print("\n")

print(len(duelist_main))
print(duelist_main["MainCode"].nunique())
print(duelist_main["MainCode"].duplicated().sum())
print("\n")

print(len(insurance))
print(insurance["MainCode"].nunique())
print(insurance["MainCode"].duplicated().sum())

26680
26680
0


26717
26717
0


757
727
30


In [106]:
insurance[insurance.duplicated("MainCode", keep=False)].head(5)

,BBB,AT,AcTypeDesc,ClientCode,MainCode,Name,InsPolicyNo,InsIssueDate,InsMaturityDate,date,InsuredAmt,InsPremium,InsurancePremium,Insurer,MortgageCode,Identifier,Phone,MobileNo,Mobile1
293,1,2C,HP-CV-FOTON-EV,29058,00102C00029058000002,RAMJI KUNWAR,KTM/CV/F/00307/081/082,2025/02/23,2026/02/22,2026-02-22,4995000.0,76598.86,0.00,IME,C8289573,BA-PRA-01-006-KHA-6158,9761845773,9.761846e+09,9.761846e+09
294,1,2C,HP-CV-FOTON-EV,29058,00102C00029058000002,RAMJI KUNWAR,KTM/PVA/F/00154/081/082,2025/02/23,2026/02/22,2026-02-22,4995000.0,7357.66,0.00,IME,C6684242,NaN,9761845773,9.761846e+09,9.761846e+09
295,1,2C,HP-CV-FOTON-EV,29285,00102C00029285000002,RAJESH SHAH,MNR/CV/R/24/25/00090,2025/03/15,2026/03/14,2026-03-14,5130000.0,85226.24,85226.24,UNITED AJOD(SELF),C4436949,SU PA PRA 02-001 JA 0498,9769832773,9.769833e+09,9.769833e+09
296,1,2C,HP-CV-FOTON-EV,29285,00102C00029285000002,RAJESH SHAH,MNR/CVA/F/24/25/00007,2025/03/15,2026/03/14,2026-03-14,5130000.0,6396.59,6396.59,UNITED AJOD(SELF),C1162629,SU PA PRA 02-001 JA 0498,9769832773,9.769833e+09,9.769833e+09
297,1,2C,HP-CV-FOTON-EV,29287,00102C00029287000002,KARN BAHADUR CHAND,MNR/CV/F/24/25/00091,2025/03/15,2026/03/14,2026-03-14,5130000.0,76226.85,76226.85,UNITED AJOD(SELF),C2279201,SU PA PRA 02-001 JA 0496,9806420268,9.806420e+09,9.806420e+09


In [107]:
duelist = duelist.drop_duplicates(subset="MainCode", keep = "last")
duelist_main = duelist_main.drop_duplicates(subset="MainCode", keep = "last")
insurance = insurance.drop_duplicates(subset="MainCode", keep = "last")

In [108]:
print(duelist['AgeingDays'].dtypes)

float64


In [109]:
duelist['AgeingDays'] = pd.to_numeric(duelist['AgeingDays'], errors='coerce')
duelist = duelist.sort_values(by='AgeingDays', ascending=False)
duelist.head(2)

,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,Address,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
18098,001,32,HP - HEAVY EQUIPMENT,01,NaN,00005629,NaN,00103200005629000002,AAYON NIRMAN SEWA PVT.LTD,"GHORAHI-10,DANG GHORAHI-10,DANG",...,JCB JS 220 TRACKED EXCAVATOR,NaN,40% Downpayment,7 HEAVY MACHINE,NaN,MID WESTERN,NaN,053,0.0,03/07/2025
18460,001,32,HP - HEAVY EQUIPMENT,01,001817,00003833,82001817041,00103200003833000004,RIDIPA CONSTRUCTION,Tewang-8 rolpa Rolpa District RAPTI ZONE,...,JCB JS205 LC Tracked Excavator,NaN,25% Downpayment,NaN,Ra 1 Ka 465,MID-WESTERN REGION,NaN,053,0.0,21/03/2025


In [110]:
pos = duelist.columns.get_loc("Name") + 1
duelist.insert(pos, "Ageing", np.nan)
duelist.insert(pos, "Loan Type", np.nan)
duelist.insert(pos, "OfficerName", np.nan)

pos = duelist.columns.get_loc("TotCharge") + 1
duelist.insert(pos, "OvDueWithInsurance", np.nan)
duelist.insert(pos, "InsurancePremium", np.nan)
duelist.insert(pos, "TotOvDue", np.nan)

pos = duelist.columns.get_loc("AgeingDays") + 1
duelist.insert(pos, "Bucket", np.nan)

pos = duelist.columns.get_loc("BranchName") + 1
duelist.insert(pos, "Dealer Name", np.nan)

In [111]:
# clean column names
duelist.columns = duelist.columns.str.strip()
duelist_main.columns = duelist_main.columns.str.strip()

# clean MainCode
duelist["MainCode"] = duelist["MainCode"].astype(str).str.strip()
duelist_main["MainCode"] = duelist_main["MainCode"].astype(str).str.strip()

cols = ["OfficerName", "Loan Type", "Dealer Name"]

df = duelist.merge(
    duelist_main[["MainCode"] + cols],
    on="MainCode",
    how="left",
    suffixes=("", "_ref")
)

for col in cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace(r'^(|None|nan|\s+)$', pd.NA, regex=True)
    df[col] = df[col].fillna(df[f"{col}_ref"])

df.drop(columns=[c + "_ref" for c in cols], inplace=True)

In [112]:
print(len(df))
df.head(2)

26680


,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,OfficerName,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
0,001,32,HP - HEAVY EQUIPMENT,01,NaN,00005629,NaN,00103200005629000002,AAYON NIRMAN SEWA PVT.LTD,Dang-Chronic Account-Mukesh Dahal-JCB,...,JCB JS 220 TRACKED EXCAVATOR,NaN,40% Downpayment,7 HEAVY MACHINE,NaN,MID WESTERN,NaN,053,0.0,03/07/2025
1,001,32,HP - HEAVY EQUIPMENT,01,001817,00003833,82001817041,00103200003833000004,RIDIPA CONSTRUCTION,Kathmandu-Chronic Account-Mukesh Dahal-JCB,...,JCB JS205 LC Tracked Excavator,NaN,25% Downpayment,NaN,Ra 1 Ka 465,MID-WESTERN REGION,NaN,053,0.0,21/03/2025


In [113]:
# insurance = insurance.rename(columns={
#     "prem":"InsurancePremium"
# })

# merge
df = df.merge(
    insurance[["MainCode", "InsurancePremium"]],
    on="MainCode",
    how="left",
    suffixes=("", "_ref")
)
# fill missing values
df["InsurancePremium"] = df["InsurancePremium"].fillna(df["InsurancePremium_ref"])

# remove extra columns
df.drop(columns=["InsurancePremium_ref"], inplace=True)

# fill remaining empty with 0
df["InsurancePremium"] = df["InsurancePremium"].fillna(0)

In [114]:
print(len(df))
print(df["MainCode"].nunique())
print(df["MainCode"].duplicated().sum())
df.head(2)

26680
26680
0


,BBB,AT,AcTypeDesc,CC,OldClientCode,ClientCode,OldAcNum,MainCode,Name,OfficerName,...,ModelDetails,LotNumber,DownPayment,VehicleModel,VehicleNo,Region,Obligor,DistrictCode,IntRateOfChgAc,ExpiryDateOfChgAc
0,001,32,HP - HEAVY EQUIPMENT,01,NaN,00005629,NaN,00103200005629000002,AAYON NIRMAN SEWA PVT.LTD,Dang-Chronic Account-Mukesh Dahal-JCB,...,JCB JS 220 TRACKED EXCAVATOR,NaN,40% Downpayment,7 HEAVY MACHINE,NaN,MID WESTERN,NaN,053,0.0,03/07/2025
1,001,32,HP - HEAVY EQUIPMENT,01,001817,00003833,82001817041,00103200003833000004,RIDIPA CONSTRUCTION,Kathmandu-Chronic Account-Mukesh Dahal-JCB,...,JCB JS205 LC Tracked Excavator,NaN,25% Downpayment,NaN,Ra 1 Ka 465,MID-WESTERN REGION,NaN,053,0.0,21/03/2025


In [115]:
age = df['AgeingDays']

conditions1 = [
    age.isna() | (age == 0),
    age <= 30,
    age <= 60,
    age <= 90,
    age <= 120,
    age <= 180,
    age <= 365,
    age > 365
]

choices1 = [
    "Regular",
    "1-30 Days",
    "31-60 Days",
    "61-90 Days",
    "91-120 Days",
    "121-180 Days",
    "181-365 Days",
    "Above 365 Days"
]

df['Ageing'] = np.select(conditions1, choices1, default="Unknown")

conditions2 = [
    age.isna() | (age == 0),
    age <= 90,
    age <= 180,
    age <= 365,
    age > 365
]

choices2 = [
    "Regular",
    "1-90 Days",
    "91-180 Days",
    "181-365 Days",
    "Above 365 Days"
]

df['Bucket'] = np.select(conditions2, choices2, default="Unknown")

In [116]:
cols = ['OutstandingBaln','IntDrAmt','PenalIntAmt','IntOnInt',
        'OvDuePrin','PastDuedInt','TotCharge']
# df[cols].dtypes
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')
df[cols].dtypes

OutstandingBaln    float64
IntDrAmt           float64
PenalIntAmt        float64
IntOnInt           float64
OvDuePrin          float64
PastDuedInt        float64
TotCharge          float64
dtype: object

In [117]:
df['TotOvDue'] = np.where(
    df['Remarks'] == "Expired",
    -df['OutstandingBaln'] + df['IntDrAmt'] + df['TotCharge'],
    df['PenalIntAmt'] + df['IntOnInt'] + df['OvDuePrin'] + df['PastDuedInt'] + df['TotCharge']
)

df['OvDueWithInsurance'] = df['TotOvDue'] + df['InsurancePremium']

In [118]:
keywords = ["sold out", "court case"]

df.loc[
    df["OfficerName"].str.contains("|".join(keywords), case=False, na=False),
    "Bucket"
] = "Above 365 Days"

In [119]:
# df.to_excel(output_file, index=False, sheet_name="Mainsheet")

In [120]:
cols = ["OfficerName", "Loan Type", "Dealer Name"]

# Prepare reference table: one row per combination
df_ref = duelist_main[["AcTypeDesc", "BranchName"] + cols].drop_duplicates(
    subset=["AcTypeDesc", "BranchName"], keep="last"
)

# Merge with df
final_df = df.merge(
    df_ref,
    on=["AcTypeDesc", "BranchName"],
    how="left",
    suffixes=("", "_ref")
)

# Fill missing values
for col in cols:
    final_df[col] = final_df[col].astype(str).str.strip().replace(r'^(|None|nan|\s+)$', pd.NA, regex=True)
    final_df[col] = final_df[col].fillna(final_df[f"{col}_ref"])

# Drop temporary reference columns
final_df.drop(columns=[f"{col}_ref" for col in cols], inplace=True)

In [121]:
final_df.to_excel(output_file, index=False, sheet_name="Mainsheet")